In [3]:
#Importar librerias
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

In [4]:
# Directorio raíz del proyecto
PROJECT_ROOT = Path.cwd().parent

# Ruta del archivo
DATA_PATH = PROJECT_ROOT / "Data" / "default of credit card clients.xls"

# Cargar dataset
df = pd.read_excel(DATA_PATH, header=1)

print("Dataset :", df.shape)

Dataset : (30000, 25)


In [5]:
# Define target variable
TARGET = "default payment next month"

# Separate features and target
X = df.drop(columns=[TARGET, "ID"])
y = df[TARGET]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (30000, 23)
Target shape: (30000,)


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nDefault proportion:")
print("Training:", round(y_train.mean() * 100, 2), "%")
print("Test:", round(y_test.mean() * 100, 2), "%")

Training set: (24000, 23)
Test set: (6000, 23)

Default proportion:
Training: 22.12 %
Test: 22.12 %


In [7]:
# Guardar SEX para futuros análisis de equidad
sex_train = X_train["SEX"].copy()
sex_test = X_test["SEX"].copy()

# Eliminar SEX de los predictores
X_train = X_train.drop(columns=["SEX"])
X_test = X_test.drop(columns=["SEX"])

# Renombrar PAY_0 a PAY_1
X_train = X_train.rename(columns={"PAY_0": "PAY_1"})
X_test = X_test.rename(columns={"PAY_0": "PAY_1"})

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

print("\nVariables:")
print(X_train.columns.tolist())

Training set: (24000, 22)
Test set: (6000, 22)

Variables:
['LIMIT_BAL', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_1', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']


In [8]:
# Agrupar categorías de EDUCATION
X_train["EDUCATION"] = X_train["EDUCATION"].replace({
    0: 4,
    5: 4,
    6: 4
})

X_test["EDUCATION"] = X_test["EDUCATION"].replace({
    0: 4,
    5: 4,
    6: 4
})

# Agrupar categorías de MARRIAGE
X_train["MARRIAGE"] = X_train["MARRIAGE"].replace({
    0: 3
})

X_test["MARRIAGE"] = X_test["MARRIAGE"].replace({
    0: 3
})

# Revisar resultados
print("EDUCATION - Training:")
print(X_train["EDUCATION"].value_counts().sort_index())

print("\nMARRIAGE - Training:")
print(X_train["MARRIAGE"].value_counts().sort_index())

EDUCATION - Training:
EDUCATION
1     8455
2    11256
3     3903
4      386
Name: count, dtype: int64

MARRIAGE - Training:
MARRIAGE
1    10892
2    12806
3      302
Name: count, dtype: int64


In [9]:
# Revisar distribución de estados de pago
pay_columns = [f"PAY_{i}" for i in range(1, 7)]

for col in pay_columns:
    print(f"\n{col}")
    print(X_train[col].value_counts().sort_index())


PAY_1
PAY_1
-2     2198
-1     4527
 0    11781
 1     3000
 2     2138
 3      245
 4       58
 5       18
 6       10
 7        9
 8       16
Name: count, dtype: int64

PAY_2
PAY_2
-2     3030
-1     4845
 0    12573
 1       25
 2     3145
 3      254
 4       78
 5       21
 6       11
 7       17
 8        1
Name: count, dtype: int64

PAY_3
PAY_3
-2     3270
-1     4733
 0    12636
 1        3
 2     3049
 3      192
 4       60
 5       16
 6       19
 7       20
 8        2
Name: count, dtype: int64

PAY_4
PAY_4
-2     3477
-1     4531
 0    13180
 1        2
 2     2542
 3      143
 4       53
 5       28
 6        3
 7       40
 8        1
Name: count, dtype: int64

PAY_5
PAY_5
-2     3659
-1     4417
 0    13560
 2     2102
 3      142
 4       65
 5       13
 6        3
 7       38
 8        1
Name: count, dtype: int64

PAY_6
PAY_6
-2     3942
-1     4576
 0    13008
 2     2230
 3      149
 4       42
 5        8
 6       11
 7       33
 8        1
Name: count, dtype: int6

In [10]:
# Crear copias para experimentar sin modificar los datos originales
X_train_cat = X_train.copy()
X_test_cat = X_test.copy()

# Convertir estados de pago a tipo categórico
pay_columns = [f"PAY_{i}" for i in range(1, 7)]

for col in pay_columns:
    X_train_cat[col] = X_train_cat[col].astype("category")
    X_test_cat[col] = X_test_cat[col].astype("category")

# Revisar tipos de datos
print(X_train_cat[pay_columns].dtypes)

PAY_1    category
PAY_2    category
PAY_3    category
PAY_4    category
PAY_5    category
PAY_6    category
dtype: object


In [11]:
# Identificar columnas monetarias
bill_columns = [f"BILL_AMT{i}" for i in range(1, 7)]
payment_columns = [f"PAY_AMT{i}" for i in range(1, 7)]

# Función para resumir distribuciones
def monetary_summary(data, columns):
    return data[columns].describe(
        percentiles=[0.50, 0.90, 0.95, 0.99, 0.999]
    ).T[
        ["mean", "std", "min", "50%", "90%", "95%", "99%", "99.9%", "max"]
    ]

print("BILL_AMT:")
display(monetary_summary(X_train, bill_columns))

print("\nPAY_AMT:")
display(monetary_summary(X_train, payment_columns))

BILL_AMT:


,mean,std,min,50%,90%,95%,99%,99.9%,max
BILL_AMT1,51100.502250,73510.032060,-165580.0,22330.0,141532.2,199801.05,348402.41,520653.967,964511.0
BILL_AMT2,49012.267583,70840.102410,-69777.0,21107.0,136618.0,193244.05,334929.25,510472.180,983931.0
BILL_AMT3,46861.880083,68403.933692,-61506.0,20031.0,132135.9,186886.00,323101.81,491956.762,855086.0
BILL_AMT4,43156.661458,64046.730878,-81334.0,19004.5,122821.1,173812.30,304594.51,479800.787,891586.0
BILL_AMT5,40164.412625,60627.850612,-81334.0,18102.5,115808.5,165512.30,284834.81,466775.797,927171.0
BILL_AMT6,38675.979875,59308.737828,-339603.0,16993.0,111983.2,161732.90,274502.88,456599.472,961664.0



PAY_AMT:


,mean,std,min,50%,90%,95%,99%,99.9%,max
PAY_AMT1,5623.556292,16148.316646,0.0,2100.5,10222.6,18094.15,67572.32,210000.000,873552.0
PAY_AMT2,5879.974917,20252.983189,0.0,2014.0,10411.3,19058.15,77010.79,258911.319,1215471.0
PAY_AMT3,5215.777583,17513.554475,0.0,1805.5,10000.0,17805.20,70077.15,213103.406,896040.0
PAY_AMT4,4790.331833,15060.693585,0.0,1500.0,9664.1,16406.40,64760.42,188844.154,621000.0
PAY_AMT5,4769.941750,15048.470405,0.0,1500.0,9475.1,16000.00,64578.20,194542.288,426529.0
PAY_AMT6,5229.905500,17850.346975,0.0,1500.0,9507.2,17500.00,83025.53,220371.629,528666.0


In [12]:
# Crear copias para el experimento
X_train_log = X_train.copy()
X_test_log = X_test.copy()

# Transformación logarítmica de los montos pagados
for col in payment_columns:
    X_train_log[col] = np.log1p(X_train_log[col])
    X_test_log[col] = np.log1p(X_test_log[col])

# Comparar distribuciones antes y después
comparison = pd.DataFrame({
    "Original Mean": X_train[payment_columns].mean(),
    "Log Mean": X_train_log[payment_columns].mean(),
    "Original Std": X_train[payment_columns].std(),
    "Log Std": X_train_log[payment_columns].std(),
    "Original Max": X_train[payment_columns].max(),
    "Log Max": X_train_log[payment_columns].max()
})

display(comparison.round(3))

,Original Mean,Log Mean,Original Std,Log Std,Original Max,Log Max
PAY_AMT1,5623.556,6.637,16148.317,3.244,873552,13.680
PAY_AMT2,5879.975,6.566,20252.983,3.281,1215471,14.011
PAY_AMT3,5215.778,6.283,17513.554,3.354,896040,13.706
PAY_AMT4,4790.332,6.075,15060.694,3.402,621000,13.339
PAY_AMT5,4769.942,6.029,15048.470,3.449,426529,12.963
PAY_AMT6,5229.906,5.918,17850.347,3.537,528666,13.178


In [13]:
# Copias para el nuevo experimento
X_train_log_clip = X_train_log.copy()
X_test_log_clip = X_test_log.copy()

# Calcular límites en entrenamiento
payment_upper_limits = {}

for col in payment_columns:
    upper_limit = X_train[col].quantile(0.999)
    payment_upper_limits[col] = upper_limit

    # Recortar los valores originales y después aplicar log1p
    X_train_log_clip[col] = np.log1p(
        X_train[col].clip(upper=upper_limit)
    )

    X_test_log_clip[col] = np.log1p(
        X_test[col].clip(upper=upper_limit)
    )

# Revisar límites calculados
print("Percentile 99.9 limits:")
for col, limit in payment_upper_limits.items():
    print(f"{col}: {limit:,.2f}")

Percentile 99.9 limits:
PAY_AMT1: 210,000.00
PAY_AMT2: 258,911.32
PAY_AMT3: 213,103.41
PAY_AMT4: 188,844.15
PAY_AMT5: 194,542.29
PAY_AMT6: 220,371.63


In [14]:
# Calcular límites de recorte para BILL_AMT
bill_limits = {}

for col in bill_columns:
    lower_limit = X_train[col].quantile(0.001)
    upper_limit = X_train[col].quantile(0.999)

    bill_limits[col] = {
        "lower": lower_limit,
        "upper": upper_limit
    }

# Mostrar límites
for col, limits in bill_limits.items():
    print(
        f"{col}: "
        f"Lower = {limits['lower']:,.2f} | "
        f"Upper = {limits['upper']:,.2f}"
    )

BILL_AMT1: Lower = -2,628.02 | Upper = 520,653.97
BILL_AMT2: Lower = -5,174.30 | Upper = 510,472.18
BILL_AMT3: Lower = -5,386.35 | Upper = 491,956.76
BILL_AMT4: Lower = -4,729.27 | Upper = 479,800.79
BILL_AMT5: Lower = -6,491.18 | Upper = 466,775.80
BILL_AMT6: Lower = -11,924.51 | Upper = 456,599.47


In [15]:
# Crear copias para el experimento
X_train_bill_clip = X_train.copy()
X_test_bill_clip = X_test.copy()

# Aplicar límites calculados exclusivamente con entrenamiento
for col in bill_columns:
    lower = bill_limits[col]["lower"]
    upper = bill_limits[col]["upper"]

    X_train_bill_clip[col] = X_train[col].clip(
        lower=lower,
        upper=upper
    )

    X_test_bill_clip[col] = X_test[col].clip(
        lower=lower,
        upper=upper
    )

# Comparar máximos y mínimos originales vs. recortados
comparison_bill = pd.DataFrame({
    "Original Min": X_train[bill_columns].min(),
    "Clipped Min": X_train_bill_clip[bill_columns].min(),
    "Original Max": X_train[bill_columns].max(),
    "Clipped Max": X_train_bill_clip[bill_columns].max()
})

display(comparison_bill.round(2))

,Original Min,Clipped Min,Original Max,Clipped Max
BILL_AMT1,-165580,-2628.02,964511,520653.97
BILL_AMT2,-69777,-5174.30,983931,510472.18
BILL_AMT3,-61506,-5386.35,855086,491956.76
BILL_AMT4,-81334,-4729.27,891586,479800.79
BILL_AMT5,-81334,-6491.18,927171,466775.80
BILL_AMT6,-339603,-11924.51,961664,456599.47


In [16]:
# Crear copias para Feature Engineering
X_train_fe = X_train.copy()
X_test_fe = X_test.copy()

# Variables de mora
pay_columns = [f"PAY_{i}" for i in range(1, 7)]

for data in [X_train_fe, X_test_fe]:

    # Suma de meses de atraso registrados
    data["TOTAL_DELAY_MONTHS"] = data[pay_columns].clip(lower=0).sum(axis=1)

    # Mayor atraso registrado
    data["MRD"] = data[pay_columns].clip(lower=0).max(axis=1)

    # Cantidad de períodos con atraso
    data["EARLIER_DELAYS_N"] = (data[pay_columns] > 0).sum(axis=1)

# Revisar nuevas variables
display(
    X_train_fe[
        pay_columns + [
            "TOTAL_DELAY_MONTHS",
            "MRD",
            "EARLIER_DELAYS_N"
        ]
    ].head(10)
)

,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,TOTAL_DELAY_MONTHS,MRD,EARLIER_DELAYS_N
22788,2,2,3,2,0,0,9,3,4
29006,1,-1,-1,-2,-2,-2,1,1,1
16950,1,2,0,0,0,0,3,2,2
22280,0,0,0,0,0,0,0,0,0
11346,1,-2,-1,0,0,-2,1,1,1
4102,-1,-1,2,0,-1,2,4,2,2
28654,-1,2,2,2,2,-2,8,2,4
26150,3,3,2,2,2,2,14,3,6
15091,0,0,0,0,0,0,0,0,0
6266,0,0,0,0,0,0,0,0,0


In [17]:
# Revisar un registro directamente
row = X_train_fe.iloc[0]

print("Estados de pago:")
print(row[pay_columns].to_dict())

print("\nVariables derivadas:")
print("TOTAL_DELAY_MONTHS:", row["TOTAL_DELAY_MONTHS"])
print("MRD:", row["MRD"])
print("EARLIER_DELAYS_N:", row["EARLIER_DELAYS_N"])

Estados de pago:
{'PAY_1': 2, 'PAY_2': 2, 'PAY_3': 3, 'PAY_4': 2, 'PAY_5': 0, 'PAY_6': 0}

Variables derivadas:
TOTAL_DELAY_MONTHS: 9
MRD: 3
EARLIER_DELAYS_N: 4


In [18]:
# Crear variables de utilización del crédito
for data in [X_train_fe, X_test_fe]:

    # Utilización del período más reciente
    data["UTIL_RATE_1"] = (
        data["BILL_AMT1"] / data["LIMIT_BAL"]
    )

    # Utilización promedio de los seis períodos
    bill_columns = [f"BILL_AMT{i}" for i in range(1, 7)]

    data["UTIL_RATE_AVG"] = (
        data[bill_columns].div(data["LIMIT_BAL"], axis=0).mean(axis=1)
    )

# Revisar resultados
display(
    X_train_fe[
        ["LIMIT_BAL", "BILL_AMT1", "UTIL_RATE_1", "UTIL_RATE_AVG"]
    ].head(10)
)

,LIMIT_BAL,BILL_AMT1,UTIL_RATE_1,UTIL_RATE_AVG
22788,160000,161771,1.011069,1.034312
29006,150000,0,0.000000,0.000059
16950,10000,10171,1.017100,0.857683
22280,220000,206879,0.940359,0.846647
11346,310000,0,0.000000,0.000098
4102,240000,5367,0.022363,0.030313
28654,160000,18195,0.113719,0.109259
26150,50000,47904,0.958080,0.934690
15091,360000,120533,0.334814,0.360388
6266,320000,324693,1.014666,0.985565


In [19]:
# Crear variables de capacidad de pago
for data in [X_train_fe, X_test_fe]:

    # Ratio del período más reciente
    data["PAY_STRENGTH_1"] = np.where(
        data["BILL_AMT1"] > 0,
        data["PAY_AMT1"] / data["BILL_AMT1"],
        np.nan
    )

    # Ratios de los seis períodos
    payment_ratios = pd.DataFrame(index=data.index)

    for i in range(1, 7):
        payment_ratios[f"RATIO_{i}"] = np.where(
            data[f"BILL_AMT{i}"] > 0,
            data[f"PAY_AMT{i}"] / data[f"BILL_AMT{i}"],
            np.nan
        )

    data["PAY_STRENGTH_AVG"] = payment_ratios.mean(axis=1)

# Revisar resultados
display(
    X_train_fe[
        [
            "BILL_AMT1",
            "PAY_AMT1",
            "PAY_STRENGTH_1",
            "PAY_STRENGTH_AVG"
        ]
    ].head(10)
)

,BILL_AMT1,PAY_AMT1,PAY_STRENGTH_1,PAY_STRENGTH_AVG
22788,161771,15000,0.092724,0.040480
29006,0,53,NaN,0.000000
16950,10171,2,0.000197,0.080701
22280,206879,9100,0.043987,0.037856
11346,0,0,NaN,0.000000
4102,5367,6959,1.296628,0.464411
28654,18195,0,0.000000,14.403974
26150,47904,2000,0.041750,0.035832
15091,120533,6000,0.049779,0.043024
6266,324693,11650,0.035880,0.035647


In [20]:
# Crear variables de tendencia
for data in [X_train_fe, X_test_fe]:

    # Tendencia de facturación
    recent_bill = data[["BILL_AMT1", "BILL_AMT2", "BILL_AMT3"]].mean(axis=1)
    previous_bill = data[["BILL_AMT4", "BILL_AMT5", "BILL_AMT6"]].mean(axis=1)

    data["BILL_TREND"] = recent_bill - previous_bill

    # Tendencia de pagos
    recent_payment = data[["PAY_AMT1", "PAY_AMT2", "PAY_AMT3"]].mean(axis=1)
    previous_payment = data[["PAY_AMT4", "PAY_AMT5", "PAY_AMT6"]].mean(axis=1)

    data["PAYMENT_TREND"] = recent_payment - previous_payment

# Revisar resultados
display(
    X_train_fe[
        [
            "BILL_AMT1",
            "BILL_AMT2",
            "BILL_AMT3",
            "BILL_TREND",
            "PAY_AMT1",
            "PAY_AMT2",
            "PAY_AMT3",
            "PAYMENT_TREND"
        ]
    ].head(10)
)

,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_TREND,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAYMENT_TREND
22788,161771,172632,168541,4316.000000,15000,0,0,-3166.666667
29006,0,53,0,17.666667,53,0,0,17.666667
16950,10171,8150,8446,691.000000,2,1281,1134,272.666667
22280,206879,211037,215139,49512.000000,9100,9000,7887,3429.000000
11346,0,0,326,156.666667,0,326,0,-291.333333
4102,5367,6959,6649,-1900.333333,6959,15,0,-255.000000
28654,18195,755,44255,7173.666667,0,43500,0,13500.000000
26150,47904,48240,48602,3028.333333,2000,2001,499,-333.333333
15091,120533,124530,128518,-10425.666667,6000,6000,8000,2260.666667
6266,324693,321110,325024,16456.333333,11650,12500,11900,1540.333333


In [21]:
# Crear variables de hábitos de pago
for data in [X_train_fe, X_test_fe]:

    # Frecuencia de cada comportamiento
    data["PAY_NEG2_COUNT"] = (data[pay_columns] == -2).sum(axis=1)
    data["PAY_NEG1_COUNT"] = (data[pay_columns] == -1).sum(axis=1)
    data["PAY_ZERO_COUNT"] = (data[pay_columns] == 0).sum(axis=1)
    data["PAY_DELAY_COUNT"] = (data[pay_columns] > 0).sum(axis=1)

    # Proporción de períodos con pago puntual
    data["PAY_FULL_PAYMENT_RATIO"] = (
        data["PAY_NEG1_COUNT"] / len(pay_columns)
    )

    # Proporción de períodos sin consumo
    data["PAY_NO_CONSUMPTION_RATIO"] = (
        data["PAY_NEG2_COUNT"] / len(pay_columns)
    )

# Revisar resultados
display(
    X_train_fe[
        pay_columns + [
            "PAY_NEG2_COUNT",
            "PAY_NEG1_COUNT",
            "PAY_ZERO_COUNT",
            "PAY_DELAY_COUNT",
            "PAY_FULL_PAYMENT_RATIO",
            "PAY_NO_CONSUMPTION_RATIO"
        ]
    ].head(10)
)

,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,PAY_NEG2_COUNT,PAY_NEG1_COUNT,PAY_ZERO_COUNT,PAY_DELAY_COUNT,PAY_FULL_PAYMENT_RATIO,PAY_NO_CONSUMPTION_RATIO
22788,2,2,3,2,0,0,0,0,2,4,0.000000,0.000000
29006,1,-1,-1,-2,-2,-2,3,2,0,1,0.333333,0.500000
16950,1,2,0,0,0,0,0,0,4,2,0.000000,0.000000
22280,0,0,0,0,0,0,0,0,6,0,0.000000,0.000000
11346,1,-2,-1,0,0,-2,2,1,2,1,0.166667,0.333333
4102,-1,-1,2,0,-1,2,0,3,1,2,0.500000,0.000000
28654,-1,2,2,2,2,-2,1,1,0,4,0.166667,0.166667
26150,3,3,2,2,2,2,0,0,0,6,0.000000,0.000000
15091,0,0,0,0,0,0,0,0,6,0,0.000000,0.000000
6266,0,0,0,0,0,0,0,0,6,0,0.000000,0.000000


In [22]:
# Crear interacción entre retraso y facturación
for data in [X_train_fe, X_test_fe]:

    data["PAY1_BILL1_INTERACTION"] = (
        data["PAY_1"] * data["BILL_AMT1"]
    )

# Revisar resultados
display(
    X_train_fe[
        [
            "PAY_1",
            "BILL_AMT1",
            "PAY1_BILL1_INTERACTION"
        ]
    ].head(10)
)

,PAY_1,BILL_AMT1,PAY1_BILL1_INTERACTION
22788,2,161771,323542
29006,1,0,0
16950,1,10171,10171
22280,0,206879,0
11346,1,0,0
4102,-1,5367,-5367
28654,-1,18195,-18195
26150,3,47904,143712
15091,0,120533,0
6266,0,324693,0


In [23]:
# Identificar las nuevas variables creadas
new_features = [
    "TOTAL_DELAY_MONTHS",
    "MRD",
    "EARLIER_DELAYS_N",
    "UTIL_RATE_1",
    "UTIL_RATE_AVG",
    "PAY_STRENGTH_1",
    "PAY_STRENGTH_AVG",
    "BILL_TREND",
    "PAYMENT_TREND",
    "PAY_NEG2_COUNT",
    "PAY_NEG1_COUNT",
    "PAY_ZERO_COUNT",
    "PAY_DELAY_COUNT",
    "PAY_FULL_PAYMENT_RATIO",
    "PAY_NO_CONSUMPTION_RATIO",
    "PAY1_BILL1_INTERACTION"
]

# Revisar dimensiones
print("Dimensiones:", X_train_fe.shape)

# Revisar valores nulos e infinitos
print("\nValores nulos:")
display(X_train_fe[new_features].isnull().sum())

print("\nValores infinitos:")
print(
    np.isinf(X_train_fe[new_features].select_dtypes(include=np.number))
    .sum()
    .sum()
)

# Resumen estadístico
display(X_train_fe[new_features].describe().T)

Dimensiones: (24000, 38)

Valores nulos:


TOTAL_DELAY_MONTHS             0
MRD                            0
EARLIER_DELAYS_N               0
UTIL_RATE_1                    0
UTIL_RATE_AVG                  0
PAY_STRENGTH_1              2080
PAY_STRENGTH_AVG             771
BILL_TREND                     0
PAYMENT_TREND                  0
PAY_NEG2_COUNT                 0
PAY_NEG1_COUNT                 0
PAY_ZERO_COUNT                 0
PAY_DELAY_COUNT                0
PAY_FULL_PAYMENT_RATIO         0
PAY_NO_CONSUMPTION_RATIO       0
PAY1_BILL1_INTERACTION         0
dtype: int64


Valores infinitos:
0


,count,mean,std,min,25%,50%,75%,max
TOTAL_DELAY_MONTHS,24000.0,1.682417,3.574867,0.000000,0.000000,0.000000,2.000000,3.600000e+01
MRD,24000.0,0.682875,1.066937,0.000000,0.000000,0.000000,2.000000,8.000000e+00
EARLIER_DELAYS_N,24000.0,0.835708,1.550828,0.000000,0.000000,0.000000,1.000000,6.000000e+00
UTIL_RATE_1,24000.0,0.423342,0.410447,-0.619892,0.021788,0.314936,0.828832,6.455300e+00
UTIL_RATE_AVG,24000.0,0.372191,0.351114,-0.232590,0.030303,0.284231,0.685880,5.364308e+00
PAY_STRENGTH_1,21920.0,1.952636,85.340188,0.000000,0.037479,0.061311,0.275715,1.145367e+04
PAY_STRENGTH_AVG,23229.0,1.814056,35.786982,0.000000,0.041274,0.085726,0.893672,4.444133e+03
BILL_TREND,24000.0,8325.865319,29552.324262,-291275.000000,-1912.750000,639.166667,12684.166667,4.324283e+05
PAYMENT_TREND,24000.0,643.043236,13176.983421,-177931.333333,-316.666667,400.000000,1532.000000,7.306887e+05
PAY_NEG2_COUNT,24000.0,0.815667,1.794317,0.000000,0.000000,0.000000,0.000000,6.000000e+00


In [24]:
# Crear una alternativa agregada para capacidad de pago
for data in [X_train_fe, X_test_fe]:

    total_payments = data[payment_columns].sum(axis=1)

    total_positive_bills = (
        data[bill_columns]
        .clip(lower=0)
        .sum(axis=1)
    )

    data["PAY_STRENGTH_TOTAL"] = (
        total_payments / total_positive_bills.where(
            total_positive_bills > 0
        )
    )

# Revisar resultados
display(
    X_train_fe[
        [
            "PAY_STRENGTH_1",
            "PAY_STRENGTH_AVG",
            "PAY_STRENGTH_TOTAL"
        ]
    ].describe().T
)

,count,mean,std,min,25%,50%,75%,max
PAY_STRENGTH_1,21920.0,1.952636,85.340188,0.0,0.037479,0.061311,0.275715,11453.666667
PAY_STRENGTH_AVG,23229.0,1.814056,35.786982,0.0,0.041274,0.085726,0.893672,4444.133333
PAY_STRENGTH_TOTAL,23229.0,0.509143,6.314785,0.0,0.042720,0.096367,0.627230,797.000000


In [56]:
# Guardar conjuntos de referencia
X_train_base = X_train.copy()
X_test_base = X_test.copy()

# Conjuntos con Feature Engineering
X_train_experiment = X_train_fe.copy()
X_test_experiment = X_test_fe.copy()

# Verificar dimensiones
print("Conjunto original:")
print("Train:", X_train_base.shape)
print("Test:", X_test_base.shape)

print("\nConjunto con Feature Engineering:")
print("Train:", X_train_experiment.shape)
print("Test:", X_test_experiment.shape)

# Verificar que las variables originales se mantienen
original_columns = X_train_base.columns.tolist()

print(
    "\n¿Se conservan todas las variables originales?",
    all(col in X_train_experiment.columns for col in original_columns)
)

print(
    "Número de variables nuevas:",
    len(X_train_experiment.columns) - len(original_columns)
)

Conjunto original:
Train: (24000, 22)
Test: (6000, 22)

Conjunto con Feature Engineering:
Train: (24000, 39)
Test: (6000, 39)

¿Se conservan todas las variables originales? True
Número de variables nuevas: 17


In [26]:
## Creacion y prueba de nuevos datos

In [27]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("Validación cruzada configurada:")
print("Número de folds:", cv.get_n_splits())
print("Estratificación: activada")
print("Random state: 42")

Validación cruzada configurada:
Número de folds: 5
Estratificación: activada
Random state: 42


In [28]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# Pipeline de Logistic Regression
lr_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

print(lr_pipeline)

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=2000, random_state=42))])


In [29]:
from sklearn.model_selection import cross_validate
import pandas as pd
import numpy as np

# Métricas de evaluación
scoring = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "recall": "recall",
    "f1": "f1",
    "precision": "precision"
}

# Ejecutar validación cruzada
lr_cv_results = cross_validate(
    estimator=lr_pipeline,
    X=X_train_base,
    y=y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1
)

# Resumir resultados
lr_summary = pd.DataFrame({
    "Metric": scoring.keys(),
    "Mean": [
        lr_cv_results[f"test_{metric}"].mean()
        for metric in scoring
    ],
    "Std": [
        lr_cv_results[f"test_{metric}"].std()
        for metric in scoring
    ]
})

display(lr_summary.round(4))

,Metric,Mean,Std
0,roc_auc,0.7251,0.0103
1,average_precision,0.5060,0.0119
2,recall,0.2456,0.0130
3,f1,0.3653,0.0165
4,precision,0.7132,0.0215


In [30]:
# Diccionario de experimentos
experiments = {
    "E0_Original": X_train_base,

    "E1_PAY_Categorical": X_train_cat,

    "E2_PAY_Log": X_train_log,

    "E3_PAY_Log_Clip": X_train_log_clip,

    "E4_BILL_Clip": X_train_bill_clip,

    "E5_Feature_Engineering": X_train_fe,

    "E6_FE_PAY_Log": X_train_fe.copy(),

    "E7_FE_BILL_Clip": X_train_fe.copy()
}

# Aplicar transformación log a PAY_AMT en E6
for col in payment_columns:
    experiments["E6_FE_PAY_Log"][col] = np.log1p(
        experiments["E6_FE_PAY_Log"][col]
    )

# Aplicar clipping a BILL_AMT en E7
for col in bill_columns:
    lower = bill_limits[col]["lower"]
    upper = bill_limits[col]["upper"]

    experiments["E7_FE_BILL_Clip"][col] = (
        experiments["E7_FE_BILL_Clip"][col]
        .clip(lower=lower, upper=upper)
    )

# Revisar dimensiones
for name, data in experiments.items():
    print(f"{name}: {data.shape}")

E0_Original: (24000, 22)
E1_PAY_Categorical: (24000, 22)
E2_PAY_Log: (24000, 22)
E3_PAY_Log_Clip: (24000, 22)
E4_BILL_Clip: (24000, 22)
E5_Feature_Engineering: (24000, 39)
E6_FE_PAY_Log: (24000, 39)
E7_FE_BILL_Clip: (24000, 39)


In [31]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd


class QuantileClipper(BaseEstimator, TransformerMixin):

    def __init__(self, lower=0.001, upper=0.999):
        self.lower = lower
        self.upper = upper

    def fit(self, X, y=None):
        self.lower_limits_ = np.quantile(
            X, self.lower, axis=0
        )
        self.upper_limits_ = np.quantile(
            X, self.upper, axis=0
        )
        return self

    def transform(self, X):
        return np.clip(
            X,
            self.lower_limits_,
            self.upper_limits_
        )

In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Identificar variables PAY
pay_columns = [f"PAY_{i}" for i in range(1, 7)]

# Identificar variables de facturación y pagos
bill_columns = [f"BILL_AMT{i}" for i in range(1, 7)]
payment_columns = [f"PAY_AMT{i}" for i in range(1, 7)]

# Preprocesamiento numérico estándar
numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Preprocesamiento de variables PAY categóricas
categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

print("Preprocesadores definidos correctamente.")

Preprocesadores definidos correctamente.


In [33]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer


def build_preprocessor(X, model_name, experiment_name):

    categorical_cols = [
        col for col in X.columns
        if col in pay_columns and experiment_name == "E1_PAY_Categorical"
    ]

    numeric_cols = [
        col for col in X.columns
        if col not in categorical_cols
    ]

    transformers = []

    # Procesamiento categórico
    if categorical_cols:
        categorical_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ])

        transformers.append(
            ("categorical", categorical_pipeline, categorical_cols)
        )

    # Procesamiento numérico
    numeric_steps = []

    if experiment_name in ["E3_PAY_Log_Clip", "E6_FE_PAY_Log"]:
        # Los pagos ya están transformados con log.
        # No aplicar clipping adicional aquí.
        pass

    if experiment_name in ["E4_BILL_Clip", "E7_FE_BILL_Clip"]:
        bill_cols_present = [
            col for col in bill_columns if col in numeric_cols
        ]

        other_numeric_cols = [
            col for col in numeric_cols if col not in bill_cols_present
        ]

        if bill_cols_present:
            bill_pipeline = Pipeline([
                ("clipper", QuantileClipper()),
                ("imputer", SimpleImputer(strategy="median"))
            ])

            transformers.append(
                ("bill", bill_pipeline, bill_cols_present)
            )

        numeric_cols = other_numeric_cols

    if numeric_cols:
        if experiment_name == "E3_PAY_Log_Clip":
            pay_cols_present = [
                col for col in payment_columns if col in numeric_cols
            ]

            other_numeric_cols = [
                col for col in numeric_cols if col not in pay_cols_present
            ]

            if pay_cols_present:
                pay_pipeline = Pipeline([
                    ("clipper", QuantileClipper()),
                    ("imputer", SimpleImputer(strategy="median"))
                ])

                transformers.append(
                    ("payments", pay_pipeline, pay_cols_present)
                )

            numeric_cols = other_numeric_cols

        if numeric_cols:
            numeric_steps = [
                ("imputer", SimpleImputer(strategy="median"))
            ]

            if model_name == "Logistic Regression":
                numeric_steps.append(
                    ("scaler", StandardScaler())
                )

            transformers.append((
                "numeric",
                Pipeline(numeric_steps),
                numeric_cols
            ))

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )

In [34]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=42
    )
}

print("Modelos preparados:")
for name in models:
    print("-", name)

Modelos preparados:
- Logistic Regression
- Decision Tree
- Random Forest
- Gradient Boosting


In [35]:
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
import pandas as pd
import numpy as np

scoring = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "recall": "recall",
    "precision": "precision",
    "f1": "f1"
}

results = []

for experiment_name, X_data in experiments.items():

    print(f"\nProcesando experimento: {experiment_name}")

    for model_name, model in models.items():

        print(f"  Evaluando: {model_name}")

        preprocessor = build_preprocessor(
            X_data,
            model_name,
            experiment_name
        )

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        scores = cross_validate(
            estimator=pipeline,
            X=X_data,
            y=y_train,
            cv=cv,
            scoring=scoring,
            n_jobs=-1,
            error_score="raise"
        )

        row = {
            "Experiment": experiment_name,
            "Model": model_name
        }

        for metric in scoring:
            row[f"{metric}_mean"] = scores[f"test_{metric}"].mean()
            row[f"{metric}_std"] = scores[f"test_{metric}"].std()

        results.append(row)

results_df = pd.DataFrame(results)

print("\nEvaluación completada.")
display(results_df.round(4))


Procesando experimento: E0_Original
  Evaluando: Logistic Regression
  Evaluando: Decision Tree
  Evaluando: Random Forest
  Evaluando: Gradient Boosting

Procesando experimento: E1_PAY_Categorical
  Evaluando: Logistic Regression
  Evaluando: Decision Tree
  Evaluando: Random Forest
  Evaluando: Gradient Boosting

Procesando experimento: E2_PAY_Log
  Evaluando: Logistic Regression
  Evaluando: Decision Tree
  Evaluando: Random Forest
  Evaluando: Gradient Boosting

Procesando experimento: E3_PAY_Log_Clip
  Evaluando: Logistic Regression
  Evaluando: Decision Tree
  Evaluando: Random Forest
  Evaluando: Gradient Boosting

Procesando experimento: E4_BILL_Clip
  Evaluando: Logistic Regression
  Evaluando: Decision Tree
  Evaluando: Random Forest
  Evaluando: Gradient Boosting

Procesando experimento: E5_Feature_Engineering
  Evaluando: Logistic Regression
  Evaluando: Decision Tree
  Evaluando: Random Forest
  Evaluando: Gradient Boosting

Procesando experimento: E6_FE_PAY_Log
  Evaluan

,Experiment,Model,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std,recall_mean,recall_std,precision_mean,precision_std,f1_mean,f1_std
0,E0_Original,Logistic Regression,0.7251,0.0103,0.5060,0.0119,0.2456,0.0130,0.7132,0.0215,0.3653,0.0165
1,E0_Original,Decision Tree,0.6137,0.0051,0.2893,0.0039,0.4153,0.0135,0.3848,0.0048,0.3994,0.0082
2,E0_Original,Random Forest,0.7651,0.0040,0.5361,0.0068,0.3763,0.0107,0.6524,0.0098,0.4772,0.0090
3,E0_Original,Gradient Boosting,0.7810,0.0052,0.5527,0.0080,0.3686,0.0059,0.6793,0.0147,0.4778,0.0072
4,E1_PAY_Categorical,Logistic Regression,0.7668,0.0055,0.5420,0.0111,0.3583,0.0091,0.6784,0.0181,0.4686,0.0050
5,E1_PAY_Categorical,Decision Tree,0.6162,0.0026,0.2917,0.0022,0.4170,0.0052,0.3894,0.0045,0.4027,0.0039
6,E1_PAY_Categorical,Random Forest,0.7668,0.0027,0.5358,0.0093,0.3722,0.0045,0.6516,0.0094,0.4737,0.0046
7,E1_PAY_Categorical,Gradient Boosting,0.7818,0.0061,0.5563,0.0094,0.3635,0.0052,0.6794,0.0135,0.4736,0.0049
8,E2_PAY_Log,Logistic Regression,0.7470,0.0067,0.5082,0.0125,0.2552,0.0075,0.6797,0.0296,0.3710,0.0107
9,E2_PAY_Log,Decision Tree,0.6135,0.0052,0.2891,0.0041,0.4153,0.0135,0.3843,0.0052,0.3992,0.0084


In [36]:
from sklearn.model_selection import cross_val_predict
from sklearn.base import clone

# Candidatos seleccionados
ks_experiments = {
    "E0_Original": X_train_base,
    "E5_Feature_Engineering": X_train_fe,
    "E7_FE_BILL_Clip": X_train_fe
}

ks_results = {}

for experiment_name, X_data in ks_experiments.items():

    preprocessor = build_preprocessor(
        X_data,
        "Gradient Boosting",
        experiment_name
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", GradientBoostingClassifier(random_state=42))
    ])

    probabilities = cross_val_predict(
        pipeline,
        X_data,
        y_train,
        cv=cv,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]

    ks_results[experiment_name] = probabilities

print("Predicciones out-of-fold generadas.")

Predicciones out-of-fold generadas.


In [37]:
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import ks_2samp
import pandas as pd

ks_comparison = []

for experiment_name, probabilities in ks_results.items():

    # Separar probabilidades según el resultado real
    prob_no_default = probabilities[y_train.to_numpy() == 0]
    prob_default = probabilities[y_train.to_numpy() == 1]

    # Calcular KS
    ks_statistic = ks_2samp(
        prob_default,
        prob_no_default
    ).statistic

    ks_comparison.append({
        "Experiment": experiment_name,
        "ROC_AUC": roc_auc_score(y_train, probabilities),
        "Average_Precision": average_precision_score(
            y_train, probabilities
        ),
        "KS": ks_statistic
    })

ks_df = pd.DataFrame(ks_comparison)

display(ks_df.round(4))

,Experiment,ROC_AUC,Average_Precision,KS
0,E0_Original,0.7809,0.5513,0.4339
1,E5_Feature_Engineering,0.7855,0.5549,0.4341
2,E7_FE_BILL_Clip,0.7854,0.5560,0.4343


In [38]:
#modelos guardados
#ahora se ajustan h

In [39]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform

param_distributions = {
    "model__n_estimators": randint(100, 501),
    "model__learning_rate": loguniform(0.01, 0.2),
    "model__max_depth": randint(2, 5),
    "model__min_samples_split": randint(2, 31),
    "model__min_samples_leaf": randint(5, 31),
    "model__subsample": uniform(0.7, 0.3)
}

In [40]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.base import clone
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    recall_score,
    precision_score,
    f1_score
)
from scipy.stats import randint, uniform, loguniform
import pandas as pd

# Experimentos seleccionados
tuning_experiments = {
    "E5_Feature_Engineering": X_train_fe,
    "E7_FE_BILL_Clip": X_train_fe
}

# Espacio de búsqueda
param_distributions = {
    "model__n_estimators": randint(100, 501),
    "model__learning_rate": loguniform(0.01, 0.2),
    "model__max_depth": randint(2, 5),
    "model__min_samples_split": randint(2, 31),
    "model__min_samples_leaf": randint(5, 31),
    "model__subsample": uniform(0.7, 0.3)
}

# Métrica principal
scoring = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "recall": "recall",
    "precision": "precision",
    "f1": "f1"
}

tuning_results = []
best_estimators = {}

for experiment_name, X_data in tuning_experiments.items():

    print(f"\nOptimizando: {experiment_name}")

    # Construir pipeline
    preprocessor = build_preprocessor(
        X_data,
        "Gradient Boosting",
        experiment_name
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", GradientBoostingClassifier(random_state=42))
    ])

    # Búsqueda aleatoria
    search = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=param_distributions,
        n_iter=40,
        scoring=scoring,
        refit="roc_auc",
        cv=cv,
        random_state=42,
        n_jobs=-1,
        verbose=1,
        return_train_score=True
    )

    search.fit(X_data, y_train)

    best_estimators[experiment_name] = search.best_estimator_

    best_index = search.best_index_

    tuning_results.append({
        "Experiment": experiment_name,
        "ROC_AUC": search.cv_results_["mean_test_roc_auc"][best_index],
        "ROC_AUC_std": search.cv_results_["std_test_roc_auc"][best_index],
        "AP": search.cv_results_["mean_test_average_precision"][best_index],
        "Recall": search.cv_results_["mean_test_recall"][best_index],
        "Precision": search.cv_results_["mean_test_precision"][best_index],
        "F1": search.cv_results_["mean_test_f1"][best_index],
        "Train_ROC_AUC": search.cv_results_["mean_train_roc_auc"][best_index],
        "Best_Params": search.best_params_
    })

    print("Mejor ROC-AUC:", round(search.best_score_, 4))
    print("Mejores parámetros:", search.best_params_)

# Resultados finales
tuning_df = pd.DataFrame(tuning_results)

display(
    tuning_df.drop(columns=["Best_Params"]).round(4)
)


Optimizando: E5_Feature_Engineering
Fitting 5 folds for each of 40 candidates, totalling 200 fits
Mejor ROC-AUC: 0.7877
Mejores parámetros: {'model__learning_rate': np.float64(0.019970999139010043), 'model__max_depth': 4, 'model__min_samples_leaf': 11, 'model__min_samples_split': 13, 'model__n_estimators': 363, 'model__subsample': np.float64(0.7103165563345655)}

Optimizando: E7_FE_BILL_Clip
Fitting 5 folds for each of 40 candidates, totalling 200 fits
Mejor ROC-AUC: 0.7878
Mejores parámetros: {'model__learning_rate': np.float64(0.019970999139010043), 'model__max_depth': 4, 'model__min_samples_leaf': 11, 'model__min_samples_split': 13, 'model__n_estimators': 363, 'model__subsample': np.float64(0.7103165563345655)}


,Experiment,ROC_AUC,ROC_AUC_std,AP,Recall,Precision,F1,Train_ROC_AUC
0,E5_Feature_Engineering,0.7877,0.0045,0.5638,0.3739,0.6752,0.4812,0.8279
1,E7_FE_BILL_Clip,0.7878,0.0045,0.5637,0.3735,0.6747,0.4808,0.8281


In [41]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import ks_2samp

best_model = best_estimators["E5_Feature_Engineering"]

optimized_probabilities = cross_val_predict(
    best_model,
    X_train_fe,
    y_train,
    cv=cv,
    method="predict_proba",
    n_jobs=-1
)[:, 1]

optimized_ks = ks_2samp(
    optimized_probabilities[y_train.to_numpy() == 1],
    optimized_probabilities[y_train.to_numpy() == 0]
).statistic

print("ROC-AUC:", round(roc_auc_score(y_train, optimized_probabilities), 4))
print("Average Precision:", round(average_precision_score(y_train, optimized_probabilities), 4))
print("KS:", round(optimized_ks, 4))

ROC-AUC: 0.7877
Average Precision: 0.5624
KS: 0.4369


## Model Selection & Experimental Decisions

### 1. Feature Engineering

Feature Engineering was retained as it provided a consistent improvement over the original feature set.

The engineered variables capture additional information related to payment delays, credit utilization, payment behavior, and historical billing patterns.

### 2. Preprocessing decisions

The following alternatives were evaluated:

- **PAY categorical encoding:** Improved Logistic Regression performance, but provided only marginal gains for Gradient Boosting.
- **PAY_AMT logarithmic transformation:** Did not produce meaningful improvements for Gradient Boosting.
- **BILL_AMT clipping:** Produced nearly identical results to the non-clipped feature set. The simpler configuration was preferred.
- **Feature Engineering without clipping:** Selected as the preferred configuration due to its slightly better Average Precision and simpler preprocessing.

### 3. Hyperparameter optimization

Gradient Boosting was optimized using RandomizedSearchCV with 40 candidate configurations and stratified 5-fold cross-validation.

Selected hyperparameters:

| Parameter | Value |
|---|---:|
| n_estimators | 363 |
| learning_rate | 0.01997 |
| max_depth | 4 |
| min_samples_leaf | 11 |
| min_samples_split | 13 |
| subsample | 0.7103 |

### 4. Selected model performance

Out-of-fold evaluation results:

| Metric | Result |
|---|---:|
| ROC-AUC | 0.7877 |
| Average Precision | 0.5624 |
| KS | 0.4369 |

### 5. Final experimental decision

The selected candidate is a Gradient Boosting Classifier trained using the engineered feature set without BILL_AMT clipping.

This configuration was selected based on its overall discrimination performance, consistency across cross-validation folds, and relative simplicity compared with alternative preprocessing configurations.

3**Validation limitation:** The original test set was previously used during exploratory model comparisons. Therefore, the current cross-validation results are considered model-selection estimates rather than a fully independent final evaluation. Further validation is required before any production deployment.

In [42]:
#Guardar modelos
import joblib
import json
from pathlib import Path

# Crear carpeta de modelos
model_path = Path("../models")
model_path.mkdir(parents=True, exist_ok=True)

# Guardar pipeline seleccionado
joblib.dump(
    best_estimators["E5_Feature_Engineering"],
    model_path / "gradient_boosting_v1.joblib"
)

# Guardar metadatos
model_metadata = {
    "model": "GradientBoostingClassifier",
    "experiment": "E5_Feature_Engineering",
    "version": "1.0",
    "metrics_oof": {
        "roc_auc": 0.7877,
        "average_precision": 0.5624,
        "ks": 0.4369
    },
    "hyperparameters": best_estimators[
        "E5_Feature_Engineering"
    ].named_steps["model"].get_params()
}

with open(model_path / "model_metadata.json", "w") as file:
    json.dump(model_metadata, file, indent=4)

print("Modelo experimental v1.0 guardado correctamente.")

Modelo experimental v1.0 guardado correctamente.


Final Pipeline — End-to-End Model

In [43]:
from sklearn.base import BaseEstimator, TransformerMixin


class FeatureEngineer(BaseEstimator, TransformerMixin):

    def __init__(self):
        self.pay_columns = [f"PAY_{i}" for i in range(1, 7)]
        self.bill_columns = [f"BILL_AMT{i}" for i in range(1, 7)]
        self.payment_columns = [f"PAY_AMT{i}" for i in range(1, 7)]

    def fit(self, X, y=None):
        self.feature_names_in_ = np.array(X.columns, dtype=object)
        return self

    def transform(self, X):

        X = X.copy()

        pay = X[self.pay_columns]
        bill = X[self.bill_columns]
        payments = X[self.payment_columns]

        # 1. Historical payment delays
        X["TOTAL_DELAY_MONTHS"] = pay.clip(lower=0).sum(axis=1)

        X["MRD"] = pay.clip(lower=0).max(axis=1)

        X["EARLIER_DELAYS_N"] = (pay > 0).sum(axis=1)

        # 2. Credit utilization
        X["UTIL_RATE_1"] = X["BILL_AMT1"] / X["LIMIT_BAL"]

        X["UTIL_RATE_AVG"] = (
            bill.div(X["LIMIT_BAL"], axis=0).mean(axis=1)
        )

        # 3. Payment strength
        X["PAY_STRENGTH_1"] = (
            X["PAY_AMT1"] / X["BILL_AMT1"].where(X["BILL_AMT1"] > 0)
        )

        payment_strength = (
            payments.to_numpy() /
            bill.where(bill > 0).to_numpy()
        )

        X["PAY_STRENGTH_AVG"] = np.nanmean(
            payment_strength, axis=1
        )

        # 4. Billing and payment trends
        X["BILL_TREND"] = (
            bill.iloc[:, :3].mean(axis=1) -
            bill.iloc[:, 3:].mean(axis=1)
        )

        X["PAYMENT_TREND"] = (
            payments.iloc[:, :3].mean(axis=1) -
            payments.iloc[:, 3:].mean(axis=1)
        )

        # 5. Payment behavior counts
        X["PAY_NEG2_COUNT"] = (pay == -2).sum(axis=1)

        X["PAY_NEG1_COUNT"] = (pay == -1).sum(axis=1)

        X["PAY_ZERO_COUNT"] = (pay == 0).sum(axis=1)

        X["PAY_DELAY_COUNT"] = (pay > 0).sum(axis=1)

        # 6. Payment behavior ratios
        X["PAY_FULL_PAYMENT_RATIO"] = (
            X["PAY_NEG1_COUNT"] / 6
        )

        X["PAY_NO_CONSUMPTION_RATIO"] = (
            X["PAY_NEG2_COUNT"] / 6
        )

        # 7. Interaction feature
        X["PAY1_BILL1_INTERACTION"] = (
            X["PAY_1"] * X["BILL_AMT1"]
        )

        # 8. Total payment strength
        positive_bills = bill.clip(lower=0).sum(axis=1)

        X["PAY_STRENGTH_TOTAL"] = (
            payments.sum(axis=1) /
            positive_bills.where(positive_bills > 0)
        )

        return X

    def get_feature_names_out(self, input_features=None):

        if input_features is None:
            input_features = self.feature_names_in_

        return np.array(
            list(input_features) + [
                "TOTAL_DELAY_MONTHS",
                "MRD",
                "EARLIER_DELAYS_N",
                "UTIL_RATE_1",
                "UTIL_RATE_AVG",
                "PAY_STRENGTH_1",
                "PAY_STRENGTH_AVG",
                "BILL_TREND",
                "PAYMENT_TREND",
                "PAY_NEG2_COUNT",
                "PAY_NEG1_COUNT",
                "PAY_ZERO_COUNT",
                "PAY_DELAY_COUNT",
                "PAY_FULL_PAYMENT_RATIO",
                "PAY_NO_CONSUMPTION_RATIO",
                "PAY1_BILL1_INTERACTION",
                "PAY_STRENGTH_TOTAL"
            ],
            dtype=object
        )


In [44]:

feature_engineer = FeatureEngineer()

X_train_fe_check = feature_engineer.fit_transform(X_train_base)

print("Original feature engineering:", X_train_fe.shape)
print("New feature engineering:", X_train_fe_check.shape)

# Compare column names and order
print(
    "Same columns and order:",
    list(X_train_fe.columns) == list(X_train_fe_check.columns)
)

# Compare numerical values
max_difference = np.nanmax(
    np.abs(
        X_train_fe.to_numpy(dtype=float) -
        X_train_fe_check.to_numpy(dtype=float)
    )
)

print("Maximum numerical difference:", max_difference)

print("\nMissing values in new output:")
print(X_train_fe_check.isna().sum().sum())


Original feature engineering: (24000, 39)
New feature engineering: (24000, 39)
Same columns and order: True
Maximum numerical difference: 0.0

Missing values in new output:
3622


C:\Users\chest\AppData\Local\Temp\ipykernel_18000\3497576313.py:47: RuntimeWarning: Mean of empty slice
  X["PAY_STRENGTH_AVG"] = np.nanmean(


In [45]:
from sklearn.pipeline import Pipeline
from sklearn.base import clone
import joblib
from pathlib import Path

# Retrieve selected experimental pipeline
selected_model = best_estimators["E5_Feature_Engineering"]

# Build complete end-to-end pipeline
final_pipeline = Pipeline([
    ("feature_engineering", FeatureEngineer()),
    ("preprocessor", clone(selected_model.named_steps["preprocessor"])),
    ("model", clone(selected_model.named_steps["model"]))
])

print(final_pipeline)

Pipeline(steps=[('feature_engineering', FeatureEngineer()),
                ('preprocessor',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['LIMIT_BAL', 'EDUCATION',
                                                   'MARRIAGE', 'AGE', 'PAY_1',
                                                   'PAY_2', 'PAY_3', 'PAY_4',
                                                   'PAY_5', 'PAY_6',
                                                   'BILL_AMT1', 'BILL_AMT2',
                                                   'BILL_AMT3', 'BILL_AMT4',
                                                   'BILL_AMT5', 'BILL_AMT6',
                                                   'PAY_AMT...
                                                   'PAY_AMT5', 'PAY

In [46]:

# Train the complete pipeline using original features
final_pipeline.fit(X_train_base, y_train)

print("Complete pipeline trained successfully.")

# Verify predictions
train_probabilities = final_pipeline.predict_proba(X_train_base)[:, 1]

print("Training predictions:", train_probabilities.shape)
print("Minimum predicted PD:", train_probabilities.min())
print("Maximum predicted PD:", train_probabilities.max())


C:\Users\chest\AppData\Local\Temp\ipykernel_18000\3497576313.py:47: RuntimeWarning: Mean of empty slice
  X["PAY_STRENGTH_AVG"] = np.nanmean(


Complete pipeline trained successfully.


C:\Users\chest\AppData\Local\Temp\ipykernel_18000\3497576313.py:47: RuntimeWarning: Mean of empty slice
  X["PAY_STRENGTH_AVG"] = np.nanmean(


Training predictions: (24000,)
Minimum predicted PD: 0.030896953981979947
Maximum predicted PD: 0.8839754528255147


In [47]:

# Predictions from the original selected model
old_probabilities = selected_model.predict_proba(X_train_fe)[:, 1]

# Predictions from the complete pipeline
new_probabilities = final_pipeline.predict_proba(X_train_base)[:, 1]

# Compare predictions
difference = np.abs(old_probabilities - new_probabilities)

print("Maximum prediction difference:", difference.max())
print("Mean prediction difference:", difference.mean())

print(
    "Predictions equivalent:",
    np.allclose(old_probabilities, new_probabilities, atol=1e-10)
)


C:\Users\chest\AppData\Local\Temp\ipykernel_18000\3497576313.py:47: RuntimeWarning: Mean of empty slice
  X["PAY_STRENGTH_AVG"] = np.nanmean(


Maximum prediction difference: 0.0
Mean prediction difference: 0.0
Predictions equivalent: True


In [48]:

import joblib
import json
from pathlib import Path
from datetime import datetime

# Create models directory
model_path = Path("../models")
model_path.mkdir(parents=True, exist_ok=True)

# Save complete pipeline
pipeline_file = model_path / "credit_risk_pipeline_v1.joblib"

joblib.dump(final_pipeline, pipeline_file)

# Save metadata
metadata = {
    "model_name": "Gradient Boosting Classifier",
    "model_version": "1.0",
    "experiment": "E5_Feature_Engineering",
    "target": "default payment next month",
    "input_features": 22,
    "engineered_features": 17,
    "total_features": 39,
    "training_records": 24000,
    "hyperparameters": {
        key: value.item() if isinstance(value, np.generic) else value
        for key, value in final_pipeline.named_steps["model"].get_params().items()
    },
    "oof_metrics": {
        "roc_auc": 0.7877,
        "average_precision": 0.5624,
        "ks": 0.4369
    },
    "pipeline_components": [
        "FeatureEngineer",
        "ColumnTransformer",
        "GradientBoostingClassifier"
    ],
    "created_at": datetime.now().isoformat()
}

metadata_file = model_path / "model_metadata_v1.json"

with open(metadata_file, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=4, ensure_ascii=False)

print("Pipeline saved:", pipeline_file.resolve())
print("Metadata saved:", metadata_file.resolve())


Pipeline saved: C:\Users\chest\OneDrive\Documentos\Portafolio\Riesgo crediticio\models\credit_risk_pipeline_v1.joblib
Metadata saved: C:\Users\chest\OneDrive\Documentos\Portafolio\Riesgo crediticio\models\model_metadata_v1.json


In [49]:
#prueba

# Load saved pipeline
loaded_pipeline = joblib.load(pipeline_file)

# Predict using original test data
test_probabilities = loaded_pipeline.predict_proba(X_test_base)[:, 1]

print("Pipeline loaded successfully.")
print("Test records:", len(test_probabilities))
print("First 10 predicted probabilities:")
print(test_probabilities[:10])

# Verify output
print("\nOutput validation:")
print("All probabilities finite:", np.isfinite(test_probabilities).all())
print("All probabilities between 0 and 1:",
      ((test_probabilities >= 0) & (test_probabilities <= 1)).all())


Pipeline loaded successfully.
Test records: 6000
First 10 predicted probabilities:
[0.11180976 0.11446684 0.15788866 0.17036555 0.04474492 0.46791809
 0.09789584 0.04237056 0.09086008 0.07008049]

Output validation:
All probabilities finite: True
All probabilities between 0 and 1: True


C:\Users\chest\AppData\Local\Temp\ipykernel_18000\3497576313.py:47: RuntimeWarning: Mean of empty slice
  X["PAY_STRENGTH_AVG"] = np.nanmean(


In [58]:

import sys
import json
import joblib
import numpy as np
import pandas as pd
import sklearn
import shap

from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
from src.feature_engineering import FeatureEngineer as ImportedFeatureEngineer

# ============================================================
# 1. PROJECT PATHS AND IMPORT
# ============================================================

PROJECT_ROOT = Path.cwd().parent
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FILE = MODEL_DIR / "credit_risk_pipeline_v1.joblib"
METADATA_FILE = MODEL_DIR / "model_metadata_v1.json"
REQUIREMENTS_FILE = MODEL_DIR / "requirements_model.txt"

# ============================================================
# 2. CHECK THE EXISTING TRAINED PIPELINE
# ============================================================

assert "feature_engineering" in final_pipeline.named_steps
assert "preprocessor" in final_pipeline.named_steps
assert "model" in final_pipeline.named_steps

# Preserve predictions from the already-trained pipeline
reference_train_probabilities = (
    final_pipeline.predict_proba(X_train_base)[:, 1]
)

reference_test_probabilities = (
    final_pipeline.predict_proba(X_test_base)[:, 1]
)

# ============================================================
# 3. REPLACE THE NOTEBOOK CLASS WITH THE IMPORTABLE CLASS
# ============================================================

feature_engineer = ImportedFeatureEngineer()
feature_engineer.fit(X_train_base)

# Check that the imported class reproduces the existing features
X_train_module = feature_engineer.transform(X_train_base)

assert list(X_train_module.columns) == list(X_train_fe.columns)

np.testing.assert_allclose(
    X_train_module.to_numpy(dtype=float),
    X_train_fe.to_numpy(dtype=float),
    rtol=1e-10,
    atol=1e-10,
    equal_nan=True
)

# Replace only the feature engineering component.
# The fitted preprocessor and Gradient Boosting model remain unchanged.
final_pipeline.set_params(
    feature_engineering=feature_engineer
)

# ============================================================
# 4. VERIFY PREDICTIONS BEFORE SAVING
# ============================================================

new_train_probabilities = (
    final_pipeline.predict_proba(X_train_base)[:, 1]
)

new_test_probabilities = (
    final_pipeline.predict_proba(X_test_base)[:, 1]
)

train_max_difference = np.max(
    np.abs(reference_train_probabilities - new_train_probabilities)
)

test_max_difference = np.max(
    np.abs(reference_test_probabilities - new_test_probabilities)
)

assert np.allclose(
    reference_train_probabilities,
    new_train_probabilities,
    rtol=1e-10,
    atol=1e-10
)

assert np.allclose(
    reference_test_probabilities,
    new_test_probabilities,
    rtol=1e-10,
    atol=1e-10
)

# ============================================================
# 5. SAVE THE COMPLETE PIPELINE
# ============================================================

joblib.dump(final_pipeline, MODEL_FILE)

# ============================================================
# 6. BUILD MODEL METADATA
# ============================================================

def make_json_serializable(value):
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    return str(value)


model = final_pipeline.named_steps["model"]

metadata = {
    "project": "Credit Risk Intelligence",
    "model_name": "GradientBoostingClassifier",
    "model_version": "1.0",
    "experiment": "E5_Feature_Engineering",
    "target": "default payment next month",
    "prediction_output": "Probability of Default (PD)",

    "input": {
        "original_features": int(X_train_base.shape[1]),
        "feature_names": list(X_train_base.columns),
        "engineered_features": 17,
        "total_features": int(X_train_fe.shape[1])
    },

    "training": {
        "training_records": int(len(X_train_base)),
        "test_records": int(len(X_test_base)),
        "split": "80/20 stratified",
        "random_state": 42,
        "cross_validation": "StratifiedKFold, 5 folds"
    },

    "oof_metrics": {
        "roc_auc": 0.7877,
        "average_precision": 0.5624,
        "ks": 0.4369
    },

    "hyperparameters": {
        key: make_json_serializable(value)
        for key, value in model.get_params().items()
    },

    "pipeline_components": [
        "src.feature_engineering.FeatureEngineer",
        "ColumnTransformer",
        "GradientBoostingClassifier"
    ],

    "software_versions": {
        "python": sys.version.split()[0],
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "joblib": version("joblib"),
        "shap": shap.__version__
    },

    "created_at_utc": datetime.now(timezone.utc).isoformat(),

    "validation_notes": [
        "Pipeline receives the original input features.",
        "Feature engineering is included in the saved pipeline.",
        "Predictions were checked against the previously trained pipeline.",
        "The original test set was previously used during exploratory model comparisons.",
        "OOF metrics are cross-validation estimates, not independent external validation."
    ]
}

with open(METADATA_FILE, "w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=4, ensure_ascii=False)

# ============================================================
# 7. SAVE CORE DEPENDENCY VERSIONS
# ============================================================

requirements = [
    f"numpy=={np.__version__}",
    f"pandas=={pd.__version__}",
    f"scikit-learn=={sklearn.__version__}",
    f"joblib=={version('joblib')}"
]

REQUIREMENTS_FILE.write_text(
    "\n".join(requirements) + "\n",
    encoding="utf-8"
)

# ============================================================
# 8. RELOAD AND TEST THE SAVED ARTIFACT
# ============================================================

loaded_pipeline = joblib.load(MODEL_FILE)

loaded_test_probabilities = (
    loaded_pipeline.predict_proba(X_test_base)[:, 1]
)

assert np.allclose(
    new_test_probabilities,
    loaded_test_probabilities,
    rtol=1e-10,
    atol=1e-10
)

assert np.isfinite(loaded_test_probabilities).all()
assert (
    (loaded_test_probabilities >= 0) &
    (loaded_test_probabilities <= 1)
).all()

# ============================================================
# 9. FINAL REPORT
# ============================================================

print("=" * 60)
print("FINAL MODEL ARTIFACT SAVED AND VERIFIED")
print("=" * 60)

print("\nPipeline:", MODEL_FILE.resolve())
print("Metadata:", METADATA_FILE.resolve())
print("Dependencies:", REQUIREMENTS_FILE.resolve())

print("\nFeatureEngineer module:",
      type(loaded_pipeline.named_steps["feature_engineering"]).__module__)

print("\nOriginal input features:", X_train_base.shape[1])
print("Engineered features:", X_train_fe.shape[1])
print("Training records:", len(X_train_base))

print("\nMaximum train prediction difference:", train_max_difference)
print("Maximum test prediction difference:", test_max_difference)

print("\nReloaded pipeline test records:", len(loaded_test_probabilities))
print("All probabilities valid:",
      np.isfinite(loaded_test_probabilities).all())

print("\nAll checks passed successfully.")




FINAL MODEL ARTIFACT SAVED AND VERIFIED

Pipeline: C:\Users\chest\OneDrive\Documentos\Portafolio\Riesgo crediticio\models\credit_risk_pipeline_v1.joblib
Metadata: C:\Users\chest\OneDrive\Documentos\Portafolio\Riesgo crediticio\models\model_metadata_v1.json
Dependencies: C:\Users\chest\OneDrive\Documentos\Portafolio\Riesgo crediticio\models\requirements_model.txt

FeatureEngineer module: src.feature_engineering

Original input features: 22
Engineered features: 39
Training records: 24000

Maximum train prediction difference: 0.0
Maximum test prediction difference: 0.0

Reloaded pipeline test records: 6000
All probabilities valid: True

All checks passed successfully.


In [60]:
from pathlib import Path
import joblib

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data" / "processed"

DATA_DIR.mkdir(parents=True, exist_ok=True)

# Guardar conjuntos de datos y etiquetas
joblib.dump(
    {
        "X_train_base": X_train_base,
        "X_test_base": X_test_base,
        "y_train": y_train,
        "y_test": y_test
    },
    DATA_DIR / "train_test_split.joblib"
)

print("Conjuntos de datos guardados correctamente.")
print("X_train:", X_train_base.shape)
print("X_test:", X_test_base.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

Conjuntos de datos guardados correctamente.
X_train: (24000, 22)
X_test: (6000, 22)
y_train: (24000,)
y_test: (6000,)
